# 09 Feature Engineering

## Objective

Objective:
Prepare model-ready features from the cleaned churn dataset using the findings from the earlier exploratory notebooks.

This step helps in:
- Converting cleaned data into modeling-friendly inputs
- Preserving strong churn signals discovered in bivariate and multivariate analysis
- Creating useful transformed and interaction features
- Preparing a consistent dataset for downstream feature selection and modeling


## Imports


In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_selection import chi2, mutual_info_classif

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import CATEGORICAL_COLUMNS, NUMERICAL_COLUMNS, TARGET_COLUMN
from src.data.data_loader import load_cleaned_data

sns.set_theme(style="whitegrid")


## Load Clean Data

Load the cleaned churn dataset and separate the target, numerical, and categorical feature groups that will be used in feature engineering.


In [3]:
df = load_cleaned_data()

target_column = TARGET_COLUMN
numerical_columns = [column for column in NUMERICAL_COLUMNS if column in df.columns]
categorical_columns = [column for column in CATEGORICAL_COLUMNS if column in df.columns]

feature_groups = {
    "target_column": target_column,
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
    "shape": df.shape,
}

feature_groups


{'target_column': 'Churn',
 'numerical_columns': ['tenure', 'MonthlyCharges', 'TotalCharges'],
 'categorical_columns': ['gender',
  'SeniorCitizen',
  'Partner',
  'Dependents',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod'],
 'shape': (7043, 21)}

## Revisit EDA Findings

Before engineering new features, summarize the main findings from notebooks `05`, `06`, and `07` so the next steps stay tied to the actual churn patterns already observed.

- From `05_univariate_analysis.ipynb`, the dataset showed uneven category sizes, meaningful structural labels such as `No internet service`, and numerical distributions that may need careful transformation instead of naive scaling.
- From `06_bivariate_analysis.ipynb`, the strongest single-feature churn signals were `tenure`, `MonthlyCharges`, `TotalCharges`, `Contract`, `PaymentMethod`, `TechSupport`, `OnlineSecurity`, and `InternetService`, while weaker features such as `gender` and `PhoneService` showed much less separation.
- From `07_multivariate_analysis.ipynb`, `tenure` and `TotalCharges` were strongly related, and high-risk combinations such as `Month-to-month + Electronic check` and `Fiber optic + No TechSupport` suggested that interaction features may add value during modeling.

These findings guide the feature-engineering stage: preserve strong churn signals, handle redundant lifecycle features carefully, retain meaningful structural categories, and consider interaction features where the combined risk pattern is sharper than the single-feature pattern.


## Handle Categorical Encoding

Encode categorical features according to their business meaning.

- Binary features are mapped to `0/1`.
- `Contract` is ordinal-encoded because it reflects increasing customer commitment from `Month-to-month` to `Two year`.
- Multi-class service and payment features are one-hot encoded so structural labels such as `No internet service` remain distinct from ordinary `No` values.


In [9]:
encoded_df = df.copy()

binary_mappings = {
    "gender": {"Female": 0, "Male": 1},
    "Partner": {"No": 0, "Yes": 1},
    "Dependents": {"No": 0, "Yes": 1},
    "PhoneService": {"No": 0, "Yes": 1},
    "PaperlessBilling": {"No": 0, "Yes": 1},
    target_column: {"No": 0, "Yes": 1},
}

for column, mapping in binary_mappings.items():
    if column in encoded_df.columns:
        encoded_df[column] = encoded_df[column].map(mapping)

if "SeniorCitizen" in encoded_df.columns:
    encoded_df["SeniorCitizen"] = encoded_df["SeniorCitizen"].astype(int)

contract_mapping = {
    "Month-to-month": 0,
    "One year": 1,
    "Two year": 2,
}
if "Contract" in encoded_df.columns:
    encoded_df["Contract"] = encoded_df["Contract"].map(contract_mapping)

one_hot_columns = [
    column
    for column in [
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "PaymentMethod",
    ]
    if column in encoded_df.columns
]

encoded_df = pd.get_dummies(
    encoded_df,
    columns=one_hot_columns,
    drop_first=False,
    dtype=int,
)

encoding_summary = {
    "binary_encoded": [column for column in binary_mappings if column in df.columns],
    "ordinal_encoded": [column for column in ["Contract"] if column in df.columns],
    "one_hot_encoded": one_hot_columns,
    "encoded_shape": encoded_df.shape,
}

encoding_summary


{'binary_encoded': ['gender',
  'Partner',
  'Dependents',
  'PhoneService',
  'PaperlessBilling',
  'Churn'],
 'ordinal_encoded': ['Contract'],
 'one_hot_encoded': ['MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'PaymentMethod'],
 'encoded_shape': (7043, 40)}

In [10]:
encoded_df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,Contract,PaperlessBilling,MonthlyCharges,...,StreamingTV_No,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,0,0,1,0,1,0,0,1,29.85,...,1,0,0,1,0,0,0,0,1,0
1,5575-GNVDE,1,0,0,0,34,1,1,0,56.95,...,1,0,0,1,0,0,0,0,0,1
2,3668-QPYBK,1,0,0,0,2,1,0,1,53.85,...,1,0,0,1,0,0,0,0,0,1
3,7795-CFOCW,1,0,0,0,45,0,1,0,42.30,...,1,0,0,1,0,0,1,0,0,0
4,9237-HQITU,0,0,0,0,2,1,0,1,70.70,...,1,0,0,1,0,0,0,0,1,0


## Numerical Feature Transformations

Apply numerical transformations where they support modeling.

- Keep original numerical columns intact.
- Create a new log-transformed column for `TotalCharges` because it is the main skewed numerical feature.
- Add new standardized columns such as `tenure_scaled` and `MonthlyCharges_scaled` for scale-sensitive models, while preserving the original values for tree-based models.


In [11]:
transformed_df = encoded_df.copy()

if "TotalCharges" in transformed_df.columns:
    transformed_df["TotalCharges_log"] = np.log1p(transformed_df["TotalCharges"])

scaling_columns = [
    column for column in ["tenure", "MonthlyCharges", "TotalCharges"]
    if column in transformed_df.columns
]

scaled_column_names = []
for column in scaling_columns:
    column_mean = transformed_df[column].mean()
    column_std = transformed_df[column].std(ddof=0)
    scaled_column_name = f"{column}_scaled"
    if column_std:
        transformed_df[scaled_column_name] = (
            transformed_df[column] - column_mean
        ) / column_std
    else:
        transformed_df[scaled_column_name] = 0.0
    scaled_column_names.append(scaled_column_name)

scaled_numerical_df = transformed_df[scaled_column_names].copy()

transformation_summary = {
    "log_transformed": [column for column in ["TotalCharges_log"] if column in transformed_df.columns],
    "scaled_columns": scaled_column_names,
    "transformed_shape": transformed_df.shape,
    "scaled_numerical_shape": scaled_numerical_df.shape,
}

transformation_summary


{'log_transformed': ['TotalCharges_log'],
 'scaled_columns': ['tenure_scaled',
  'MonthlyCharges_scaled',
  'TotalCharges_scaled'],
 'transformed_shape': (7043, 44),
 'scaled_numerical_shape': (7043, 3)}

In [9]:
transformed_df[[column for column in ["TotalCharges", "TotalCharges_log"] if column in transformed_df.columns]].head()


,TotalCharges,TotalCharges_log
0,29.85,3.429137
1,1889.50,7.544597
2,108.15,4.692723
3,1840.75,7.518471
4,151.65,5.028148


In [12]:
transformed_df[[column for column in scaling_columns + scaled_column_names if column in transformed_df.columns]].head()


,tenure,MonthlyCharges,TotalCharges,tenure_scaled,MonthlyCharges_scaled,TotalCharges_scaled
0,1,29.85,29.85,-1.277445,-1.160323,-0.992611
1,34,56.95,1889.50,0.066327,-0.259629,-0.172165
2,2,53.85,108.15,-1.236724,-0.362660,-0.958066
3,45,42.30,1840.75,0.514251,-0.746535,-0.193672
4,2,70.70,151.65,-1.236724,0.197365,-0.938874


## Create Interaction Features

Create interaction features where churn risk became sharper when variables were combined in multivariate analysis.

- `tenure x MonthlyCharges` captures the joint effect of customer age and monthly bill level.
- `Contract x PaymentMethod` captures risk patterns such as `Month-to-month + Electronic check`.
- `InternetService x TechSupport` and `InternetService x OnlineSecurity` preserve the high-risk service combinations observed in the interaction heatmaps.


In [13]:
interaction_df = transformed_df.copy()

if {"tenure", "MonthlyCharges"}.issubset(interaction_df.columns):
    interaction_df["tenure_x_MonthlyCharges"] = (
        interaction_df["tenure"] * interaction_df["MonthlyCharges"]
    )

payment_method_columns = [
    column for column in interaction_df.columns
    if column.startswith("PaymentMethod_")
]
for column in payment_method_columns:
    interaction_df[f"Contract_x_{column}"] = interaction_df["Contract"] * interaction_df[column]

internet_service_columns = [
    column for column in interaction_df.columns
    if column.startswith("InternetService_")
]
tech_support_columns = [
    column for column in interaction_df.columns
    if column.startswith("TechSupport_")
]
online_security_columns = [
    column for column in interaction_df.columns
    if column.startswith("OnlineSecurity_")
]

for internet_column in internet_service_columns:
    for support_column in tech_support_columns:
        interaction_df[f"{internet_column}_x_{support_column}"] = (
            interaction_df[internet_column] * interaction_df[support_column]
        )

for internet_column in internet_service_columns:
    for security_column in online_security_columns:
        interaction_df[f"{internet_column}_x_{security_column}"] = (
            interaction_df[internet_column] * interaction_df[security_column]
        )

interaction_columns = [
    column for column in interaction_df.columns
    if "_x_" in column
]

interaction_summary = {
    "interaction_count": len(interaction_columns),
    "sample_interactions": interaction_columns[:10],
    "interaction_shape": interaction_df.shape,
}

interaction_summary


{'interaction_count': 23,
 'sample_interactions': ['tenure_x_MonthlyCharges',
  'Contract_x_PaymentMethod_Bank transfer (automatic)',
  'Contract_x_PaymentMethod_Credit card (automatic)',
  'Contract_x_PaymentMethod_Electronic check',
  'Contract_x_PaymentMethod_Mailed check',
  'InternetService_DSL_x_TechSupport_No',
  'InternetService_DSL_x_TechSupport_No internet service',
  'InternetService_DSL_x_TechSupport_Yes',
  'InternetService_Fiber optic_x_TechSupport_No',
  'InternetService_Fiber optic_x_TechSupport_No internet service'],
 'interaction_shape': (7043, 67)}

In [14]:
interaction_preview_columns = [
    column for column in [
        "tenure",
        "MonthlyCharges",
        "tenure_x_MonthlyCharges",
        "Contract",
        "PaymentMethod_Electronic check",
        "Contract_x_PaymentMethod_Electronic check",
        "InternetService_Fiber optic",
        "TechSupport_No",
        "InternetService_Fiber optic_x_TechSupport_No",
        "OnlineSecurity_No",
        "InternetService_Fiber optic_x_OnlineSecurity_No",
    ]
    if column in interaction_df.columns
]

interaction_df[interaction_preview_columns].head()


,tenure,MonthlyCharges,tenure_x_MonthlyCharges,Contract,PaymentMethod_Electronic check,Contract_x_PaymentMethod_Electronic check,InternetService_Fiber optic,TechSupport_No,InternetService_Fiber optic_x_TechSupport_No,OnlineSecurity_No,InternetService_Fiber optic_x_OnlineSecurity_No
0,1,29.85,29.85,0,1,0,0,1,0,1,0
1,34,56.95,1936.30,1,0,0,0,1,0,0,0
2,2,53.85,107.70,0,0,0,0,1,0,0,0
3,45,42.30,1903.50,1,0,0,0,0,0,0,0
4,2,70.70,141.40,0,1,0,1,1,1,1,1


## Create Business Features

Create business-oriented features that summarize customer behavior more directly than the raw columns alone.

- `avg_monthly_spend` approximates customer spend intensity using `TotalCharges / tenure`.
- `is_new_customer` marks early-lifecycle customers, who were consistently higher-risk in earlier analysis.
- `has_support_services`, `is_month_to_month`, and `is_auto_payment` turn key business states into clear binary indicators.
- `service_count` summarizes how many service add-ons a customer uses.


In [15]:
business_df = interaction_df.copy()

if {"TotalCharges", "tenure"}.issubset(business_df.columns):
    tenure_nonzero = business_df["tenure"].replace(0, np.nan)
    business_df["avg_monthly_spend"] = (
        business_df["TotalCharges"] / tenure_nonzero
    ).fillna(business_df.get("MonthlyCharges", 0))

if "tenure" in business_df.columns:
    business_df["is_new_customer"] = (business_df["tenure"] < 12).astype(int)

support_columns = [
    column for column in ["TechSupport_Yes", "OnlineSecurity_Yes", "OnlineBackup_Yes"]
    if column in business_df.columns
]
if support_columns:
    business_df["has_support_services"] = (
        business_df[support_columns].sum(axis=1) > 0
    ).astype(int)

if "Contract" in business_df.columns:
    business_df["is_month_to_month"] = (business_df["Contract"] == 0).astype(int)

auto_payment_columns = [
    column for column in [
        "PaymentMethod_Bank transfer (automatic)",
        "PaymentMethod_Credit card (automatic)",
    ]
    if column in business_df.columns
]
if auto_payment_columns:
    business_df["is_auto_payment"] = business_df[auto_payment_columns].max(axis=1)

service_count_columns = [
    column for column in [
        "OnlineSecurity_Yes",
        "OnlineBackup_Yes",
        "DeviceProtection_Yes",
        "TechSupport_Yes",
        "StreamingTV_Yes",
        "StreamingMovies_Yes",
        "MultipleLines_Yes",
    ]
    if column in business_df.columns
]
if service_count_columns:
    business_df["service_count"] = business_df[service_count_columns].sum(axis=1)

business_feature_columns = [
    column for column in [
        "avg_monthly_spend",
        "is_new_customer",
        "has_support_services",
        "is_month_to_month",
        "is_auto_payment",
        "service_count",
    ]
    if column in business_df.columns
]

business_feature_summary = {
    "business_features": business_feature_columns,
    "business_shape": business_df.shape,
}

business_feature_summary


{'business_features': ['avg_monthly_spend',
  'is_new_customer',
  'has_support_services',
  'is_month_to_month',
  'is_auto_payment',
  'service_count'],
 'business_shape': (7043, 73)}

In [16]:
business_df[[column for column in business_feature_columns if column in business_df.columns]].head()


,avg_monthly_spend,is_new_customer,has_support_services,is_month_to_month,is_auto_payment,service_count
0,29.850000,1,1,1,0,1
1,55.573529,0,1,0,0,2
2,54.075000,1,1,1,0,2
3,40.905556,0,1,0,1,3
4,75.825000,1,0,1,0,0


## Reduce Redundancy

The main redundancy concern in this dataset is the strong overlap between `tenure` and `TotalCharges`, because both describe customer lifecycle from different angles. `tenure` captures how long the customer has stayed, while `TotalCharges` captures accumulated value over that time.

At this stage, both features are retained because they still carry useful business meaning and may perform differently across models. However, `avg_monthly_spend` is also created as a derived feature to represent spending intensity more directly. This reduces reliance on raw cumulative spend alone and gives later models a cleaner alternative to test.

The practical decision is:
- keep `tenure`
- keep `TotalCharges`
- keep `avg_monthly_spend`
- revisit whether `TotalCharges` should be dropped later if a linear model shows instability or multicollinearity problems


## Feature Selection Support

Use simple filter-style methods to rank the engineered features before formal modeling. This section is meant to support feature choice, not to replace leakage-safe feature selection on the training set later.


In [17]:
modeling_df = business_df.copy()

drop_columns = [column for column in ["customerID"] if column in modeling_df.columns]
modeling_df = modeling_df.drop(columns=drop_columns)

y = modeling_df[target_column].astype(int)
X = modeling_df.drop(columns=[target_column])

feature_matrix_summary = {
    "X_shape": X.shape,
    "y_shape": y.shape,
    "feature_count": X.shape[1],
}

feature_matrix_summary


{'X_shape': (7043, 71), 'y_shape': (7043,), 'feature_count': 71}

In [18]:
correlation_rows = []
for column in X.columns:
    correlation_value = pd.Series(X[column]).corr(y)
    correlation_rows.append(
        {
            "feature": column,
            "target_correlation": round(float(correlation_value), 4),
            "abs_target_correlation": round(abs(float(correlation_value)), 4),
        }
    )

correlation_rank_df = pd.DataFrame(correlation_rows).sort_values(
    "abs_target_correlation", ascending=False
)
display(correlation_rank_df.head(15))


/Users/mohammadmubashir/VCode/Churn/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/mohammadmubashir/VCode/Churn/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,feature,target_correlation,abs_target_correlation
68,is_month_to_month,0.4051,0.4051
6,Contract,-0.3967,0.3967
59,InternetService_Fiber optic_x_OnlineSecurity_No,0.3549,0.3549
4,tenure,-0.3522,0.3522
39,tenure_scaled,-0.3522,0.3522
50,InternetService_Fiber optic_x_TechSupport_No,0.3520,0.3520
16,OnlineSecurity_No,0.3426,0.3426
25,TechSupport_No,0.3373,0.3373
66,is_new_customer,0.3177,0.3177
14,InternetService_Fiber optic,0.3080,0.3080


In [19]:
chi2_scores, chi2_p_values = chi2(X.clip(lower=0), y)
chi2_rank_df = pd.DataFrame(
    {
        "feature": X.columns,
        "chi2_score": np.round(chi2_scores, 4),
        "chi2_p_value": chi2_p_values,
    }
).sort_values("chi2_score", ascending=False)
display(chi2_rank_df.head(15))


,feature,chi2_score,chi2_p_value
42,tenure_x_MonthlyCharges,624391.1767,0.000000e+00
9,TotalCharges,624292.0030,0.000000e+00
4,tenure,16278.9237,0.000000e+00
8,MonthlyCharges,3680.7877,0.000000e+00
65,avg_monthly_spend,3673.5800,0.000000e+00
6,Contract,1115.7802,1.227941e-244
59,InternetService_Fiber optic_x_OnlineSecurity_No,602.9261,3.866813e-133
50,InternetService_Fiber optic_x_TechSupport_No,596.4782,9.768597e-132
68,is_month_to_month,519.8953,4.459832e-115
66,is_new_customer,501.9346,3.606069e-111


In [20]:
mutual_info_scores = mutual_info_classif(X, y, random_state=42)
mutual_info_rank_df = pd.DataFrame(
    {
        "feature": X.columns,
        "mutual_information": np.round(mutual_info_scores, 4),
    }
).sort_values("mutual_information", ascending=False)
display(mutual_info_rank_df.head(15))


,feature,mutual_information
68,is_month_to_month,0.0960
6,Contract,0.0939
4,tenure,0.0794
39,tenure_scaled,0.0704
59,InternetService_Fiber optic_x_OnlineSecurity_No,0.0606
16,OnlineSecurity_No,0.0600
25,TechSupport_No,0.0558
19,OnlineBackup_No,0.0544
50,InternetService_Fiber optic_x_TechSupport_No,0.0536
42,tenure_x_MonthlyCharges,0.0519


In [21]:
feature_selection_support_df = (
    correlation_rank_df.merge(chi2_rank_df, on="feature", how="outer")
    .merge(mutual_info_rank_df, on="feature", how="outer")
)

feature_selection_support_df = feature_selection_support_df.sort_values(
    ["mutual_information", "abs_target_correlation"],
    ascending=[False, False],
)

feature_selection_tables_dir = project_root / "reports" / "tables" / "feature_engineering"
feature_selection_tables_dir.mkdir(parents=True, exist_ok=True)

feature_selection_support_df.to_csv(
    feature_selection_tables_dir / "feature_selection_support.csv",
    index=False,
)

display(feature_selection_support_df.head(25))


,feature,target_correlation,abs_target_correlation,chi2_score,chi2_p_value,mutual_information
65,is_month_to_month,0.4051,0.4051,519.8953,4.459832e-115,0.0960
0,Contract,-0.3967,0.3967,1115.7802,1.227941e-244,0.0939
68,tenure,-0.3522,0.3522,16278.9237,0.000000e+00,0.0794
69,tenure_scaled,-0.3522,0.3522,467.6582,1.035985e-103,0.0704
17,InternetService_Fiber optic_x_OnlineSecurity_No,0.3549,0.3549,602.9261,3.866813e-133,0.0606
38,OnlineSecurity_No,0.3426,0.3426,416.1829,1.653059e-92,0.0600
55,TechSupport_No,0.3373,0.3373,406.1171,2.566524e-90,0.0558
35,OnlineBackup_No,0.2680,0.2680,284.0749,9.719113e-64,0.0544
20,InternetService_Fiber optic_x_TechSupport_No,0.3520,0.3520,596.4782,9.768597e-132,0.0536
70,tenure_x_MonthlyCharges,-0.1985,0.1985,624391.1767,0.000000e+00,0.0519


This feature-selection support table gives an exploratory ranking of the engineered features using multiple simple criteria. It helps identify which features look strongest, which engineered columns are adding useful signal, and which overlapping features may need a closer review during modeling.


## Final Summary

The feature-engineering process preserved the strongest original churn signals, added transformed numerical features, created interaction features from the highest-risk multivariate combinations, and introduced business-oriented features such as `avg_monthly_spend`, `is_new_customer`, and `is_auto_payment`. Across the exploratory selection checks, the strongest signals remain concentrated around contract commitment, tenure, support-related services, and high-risk interaction profiles such as `Month-to-month + Electronic check` and `Fiber optic` customers without support or security add-ons.

At this stage, the notebook has produced a richer feature set for downstream modeling while preserving the original columns for comparison. The next step is to carry these feature definitions into a leakage-safe modeling pipeline, where train-test split, fitted preprocessing, and final model-based feature selection can be handled correctly.
